In [1]:
import numpy as np
np.random.seed(0)

# Implementing the graph attention layer in NumPy

In [6]:
# adjency matrix
A = np.array([
    [1, 1, 1, 1],
    [1, 1, 0, 0],
    [1, 0, 1, 1],
    [1, 0, 1, 1]
])

A

array([[1, 1, 1, 1],
       [1, 1, 0, 0],
       [1, 0, 1, 1],
       [1, 0, 1, 1]])

In [7]:
# node features matrix
X = np.random.uniform(-1,1,(4,4))
X

array([[-0.08769934,  0.1368679 , -0.9624204 ,  0.23527099],
       [ 0.22419145,  0.23386799,  0.88749616,  0.3636406 ],
       [-0.2809842 , -0.12593609,  0.39526239, -0.87954906],
       [ 0.33353343,  0.34127574, -0.57923488, -0.7421474 ]])

In [10]:
# define weight matrix
W = np.random.uniform(-1, 1, (2, 4))
W

array([[ 0.30621665, -0.49341679, -0.06737845, -0.51114882],
       [-0.68206083, -0.77924972,  0.31265918, -0.7236341 ]])

In [12]:
W_att = np.random.uniform(-1, 1, (1, 4))
W_att

array([[ 0.67588981, -0.80780318,  0.95291893, -0.0626976 ]])

In [14]:
connections = np.where(A > 0)
connections

(array([0, 0, 0, 0, 1, 1, 2, 2, 2, 3, 3, 3]),
 array([0, 1, 2, 3, 0, 1, 0, 2, 3, 0, 2, 3]))

In [15]:
np.concatenate([(X @ W.T)[connections[0]], (X @ W.T)
[connections[1]]], axis=1)

array([[-0.14980001, -0.51799767, -0.14980001, -0.51799767],
       [-0.14980001, -0.51799767, -0.29241582, -0.32081269],
       [-0.14980001, -0.51799767,  0.39904523,  1.04983808],
       [-0.14980001, -0.51799767,  0.35211803, -0.13748905],
       [-0.29241582, -0.32081269, -0.14980001, -0.51799767],
       [-0.29241582, -0.32081269, -0.29241582, -0.32081269],
       [ 0.39904523,  1.04983808, -0.14980001, -0.51799767],
       [ 0.39904523,  1.04983808,  0.39904523,  1.04983808],
       [ 0.39904523,  1.04983808,  0.35211803, -0.13748905],
       [ 0.35211803, -0.13748905, -0.14980001, -0.51799767],
       [ 0.35211803, -0.13748905,  0.39904523,  1.04983808],
       [ 0.35211803, -0.13748905,  0.35211803, -0.13748905]])

In [17]:
a = W_att @ np.concatenate([(X @ W.T)[connections[0]], (X
@ W.T)[connections[1]]], axis=1).T
a

array([[ 0.20692182,  0.05865748,  0.6316273 ,  0.66135204, -0.04875742,
        -0.19702175, -0.68862199, -0.26391651, -0.23419177,  0.23878702,
         0.66349251,  0.69321724]])

In [20]:
def leaky_relu(x, alpha=0.2):
    return np.maximum(alpha*x, x)
e = leaky_relu(a)
e

array([[ 0.20692182,  0.05865748,  0.6316273 ,  0.66135204, -0.00975148,
        -0.03940435, -0.1377244 , -0.0527833 , -0.04683835,  0.23878702,
         0.66349251,  0.69321724]])

In [21]:
E = np.zeros(A.shape)
E[connections[0], connections[1]] = e[0]
E

array([[ 0.20692182,  0.05865748,  0.6316273 ,  0.66135204],
       [-0.00975148, -0.03940435,  0.        ,  0.        ],
       [-0.1377244 ,  0.        , -0.0527833 , -0.04683835],
       [ 0.23878702,  0.        ,  0.66349251,  0.69321724]])

In [22]:
def softmax2D(x, axis):
    e = np.exp(x - np.expand_dims(np.max(x, axis=axis),
    axis))
    sum = np.expand_dims(np.sum(e, axis=axis), axis)
    return e / sum
W_alpha = softmax2D(E, 1)
W_alpha

array([[0.20134422, 0.17359963, 0.30788351, 0.31717264],
       [0.25060265, 0.24328066, 0.25305835, 0.25305835],
       [0.23086923, 0.2649592 , 0.25133647, 0.2528351 ],
       [0.20441545, 0.16099405, 0.31257984, 0.32201066]])

In [23]:
H = A.T @ W_alpha @ X @ W.T
H

array([[0.4727063 , 0.29350785],
       [0.23502575, 0.14264969],
       [0.39129755, 0.27048942],
       [0.39129755, 0.27048942]])

# Implementing a GAT in PyTorch Geometric

In [31]:
import numpy as np
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import LabelEncoder

# 1) Load node features & labels
content = np.loadtxt('data/Cora/cora.content', dtype=str)
ids, feats, labels = content[:,0], content[:,1:-1].astype(float), content[:,-1]
le = LabelEncoder().fit(labels)
y = le.transform(labels)

# map paper_id → index
id2idx = {id_:i for i,id_ in enumerate(ids)}

# 2) Load edges
edges = np.loadtxt('data/Cora/cora.cites', dtype=str)
src = [id2idx[s] for s in edges[:,0]]
dst = [id2idx[d] for d in edges[:,1]]
edge_index = torch.tensor([src+dst, dst+src], dtype=torch.long)  # undirected

# 3) Build Data
x = torch.tensor(feats, dtype=torch.float)
y = torch.tensor(y,     dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y)

# 4) Create train/val/test masks however you like, e.g. 140/500/1000 splits:
num_nodes = x.size(0)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[:140] = True
# … similarly for val_mask, test_mask …

data.train_mask = train_mask
data.val_mask   = val_mask
data.test_mask  = test_mask

print(data)

FileNotFoundError: data/Cora/cora.content not found.